# Object Detection Inference Optimization Pipeline
### SP26 Homework 2 — Raeeka Yusuf (018233761)

**Option 2: Inference Optimization**

1. **Model Export** — YOLOv8m + RT-DETR-L → ONNX + TensorRT
2. **Benchmarking** — Latency + mAP on our own annotated data
3. **Backend Server** — FastAPI + ngrok for React frontend

**Setup:** Runtime → Change runtime type → **T4 GPU** → Save

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics onnxruntime-gpu pycocotools fastapi uvicorn python-multipart pyngrok

In [ ]:
import torch
from ultralytics import YOLO, RTDETR
from pathlib import Path
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

---
# Part 0: Load Our Own Data from Google Drive

**Before running this:** Upload your images and video to Google Drive:
```
My Drive/Object-Detection-Data/images/   ← all your photos (.jpg, .png)
My Drive/Object-Detection-Data/video.mp4  ← your video
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, glob, os

# Where your data lives on Drive
DRIVE_FOLDER = "/content/drive/MyDrive/Object-Detection-Data"

# Local working directories
IMG_DIR = Path("/content/my_dataset/images")
LABEL_DIR = Path("/content/my_dataset/labels")
IMG_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Copy images from Drive to local (faster I/O)
drive_images = glob.glob(f"{DRIVE_FOLDER}/images/*")
for src in drive_images:
    shutil.copy2(src, IMG_DIR / Path(src).name)

image_files = sorted(IMG_DIR.glob("*"))
print(f"Copied {len(image_files)} images to {IMG_DIR}")

# Copy video
video_files = glob.glob(f"{DRIVE_FOLDER}/*.mp4") + glob.glob(f"{DRIVE_FOLDER}/*.mov")
VIDEO_PATH = None
if video_files:
    VIDEO_PATH = f"/content/my_video{Path(video_files[0]).suffix}"
    shutil.copy2(video_files[0], VIDEO_PATH)
    print(f"Copied video to {VIDEO_PATH}")
else:
    print("No video found — add a .mp4 or .mov to your Drive folder")

---
# Part 1: Model Export

In [ ]:
# Load models
yolo_model = YOLO("yolov8m.pt")
rtdetr_model = RTDETR("rtdetr-l.pt")
print(f"YOLOv8m: {sum(p.numel() for p in yolo_model.model.parameters())/1e6:.1f}M params")
print(f"RT-DETR-L: {sum(p.numel() for p in rtdetr_model.model.parameters())/1e6:.1f}M params")

## 1.1 Sanity check — PyTorch inference on our own image

In [ ]:
from IPython.display import display, Image as IPImage
import cv2, numpy as np

# Use first image from our dataset
sample_path = str(image_files[0])

yolo_results = yolo_model(sample_path)
rtdetr_results = rtdetr_model(sample_path)

for name, results in [("YOLOv8", yolo_results), ("RT-DETR", rtdetr_results)]:
    annotated = results[0].plot()
    out_path = f"sample_{name.lower().replace('-','')}.jpg"
    cv2.imwrite(out_path, annotated)
    print(f"\n{name} — {len(results[0].boxes)} detections:")
    display(IPImage(filename=out_path, width=500))

## 1.2 Export to ONNX

In [ ]:
yolo_onnx_path = yolo_model.export(format="onnx", simplify=True, dynamic=False)
print(f"YOLOv8 ONNX: {yolo_onnx_path} ({Path(yolo_onnx_path).stat().st_size/1e6:.1f} MB)")

In [ ]:
rtdetr_for_onnx = RTDETR("rtdetr-l.pt")  # fresh load avoids CUDA buffer issues
rtdetr_onnx_path = rtdetr_for_onnx.export(format="onnx", simplify=True, dynamic=False)
print(f"RT-DETR ONNX: {rtdetr_onnx_path} ({Path(rtdetr_onnx_path).stat().st_size/1e6:.1f} MB)")

## 1.3 Verify ONNX exports

In [ ]:
yolo_onnx = YOLO(yolo_onnx_path)
rtdetr_onnx = RTDETR(rtdetr_onnx_path)

r1 = yolo_onnx(sample_path, verbose=False)
r2 = rtdetr_onnx(sample_path, verbose=False)
print(f"YOLOv8  — PyTorch: {len(yolo_results[0].boxes)}, ONNX: {len(r1[0].boxes)}")
print(f"RT-DETR — PyTorch: {len(rtdetr_results[0].boxes)}, ONNX: {len(r2[0].boxes)}")

## 1.4 Export to TensorRT (FP16)

Takes ~5-10 min per model. Skips if engine already exists.

In [ ]:
yolo_trt_path = "yolov8m.engine"
if Path(yolo_trt_path).exists():
    print(f"Already exists: {yolo_trt_path}")
else:
    yolo_trt_path = yolo_model.export(format="engine", half=True, dynamic=False)
    print(f"YOLOv8 TensorRT: {yolo_trt_path}")
print(f"Size: {Path(yolo_trt_path).stat().st_size/1e6:.1f} MB")

In [ ]:
rtdetr_trt_path = "rtdetr-l.engine"
if Path(rtdetr_trt_path).exists():
    print(f"Already exists: {rtdetr_trt_path}")
else:
    rtdetr_for_trt = RTDETR("rtdetr-l.pt")
    rtdetr_trt_path = rtdetr_for_trt.export(format="engine", half=True, dynamic=False)
    print(f"RT-DETR TensorRT: {rtdetr_trt_path}")
print(f"Size: {Path(rtdetr_trt_path).stat().st_size/1e6:.1f} MB")

## 1.5 Verify TensorRT exports

In [ ]:
import pandas as pd

yolo_trt = YOLO(yolo_trt_path)
rtdetr_trt = RTDETR(rtdetr_trt_path)

r3 = yolo_trt(sample_path, verbose=False)
r4 = rtdetr_trt(sample_path, verbose=False)

comparison = pd.DataFrame({
    "Model": ["YOLOv8"]*3 + ["RT-DETR"]*3,
    "Backend": ["PyTorch","ONNX","TensorRT"]*2,
    "Detections": [len(yolo_results[0].boxes), len(r1[0].boxes), len(r3[0].boxes),
                   len(rtdetr_results[0].boxes), len(r2[0].boxes), len(r4[0].boxes)]
})
print(comparison.to_string(index=False))

In [ ]:
import os, json

model_paths = {
    "yolov8m_pytorch": "yolov8m.pt",
    "yolov8m_onnx": yolo_onnx_path,
    "yolov8m_tensorrt": yolo_trt_path,
    "rtdetr_l_pytorch": "rtdetr-l.pt",
    "rtdetr_l_onnx": rtdetr_onnx_path,
    "rtdetr_l_tensorrt": rtdetr_trt_path,
}
with open("model_paths.json", "w") as f:
    json.dump(model_paths, f, indent=2)
print("Saved model_paths.json")

---
# Part 2: Create Our Own Annotations

We use **model-assisted annotation**: run YOLOv8 PyTorch on our images at high confidence, then manually review. This generates YOLO-format label files for mAP evaluation.

In [ ]:
# Auto-annotate all our images using YOLOv8 at high confidence
# High confidence (0.5) = fewer false positives = cleaner ground truth
print(f"Auto-annotating {len(image_files)} images...")

annotator = YOLO("yolov8m.pt")
total_annotations = 0

for img_path in image_files:
    results = annotator(str(img_path), conf=0.5, verbose=False)
    boxes = results[0].boxes
    
    # Save in YOLO format: class_id cx cy w h (normalized)
    label_path = LABEL_DIR / f"{img_path.stem}.txt"
    ih, iw = results[0].orig_shape
    
    with open(label_path, "w") as f:
        for box in boxes:
            cls_id = int(box.cls[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = ((x1 + x2) / 2) / iw
            cy = ((y1 + y2) / 2) / ih
            bw = (x2 - x1) / iw
            bh = (y2 - y1) / ih
            f.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
            total_annotations += 1

print(f"Done! {total_annotations} annotations across {len(image_files)} images")
print(f"Labels saved to {LABEL_DIR}")

In [ ]:
# Review: show 6 annotated images so you can visually verify
import matplotlib.pyplot as plt
import random

review_imgs = random.sample(image_files, min(6, len(image_files)))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, img_path in enumerate(review_imgs):
    results = annotator(str(img_path), conf=0.5, verbose=False)
    annotated = results[0].plot()
    axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[i].set_title(img_path.name, fontsize=9)
    axes[i].axis("off")

for j in range(i+1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Review: These are your ground truth annotations (conf>0.5)\nIf any are clearly wrong, note the image name", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Create data.yaml for Ultralytics val()
# COCO class names (80 classes)
coco_names = annotator.names  # dict {0: 'person', 1: 'bicycle', ...}

yaml_content = f"""path: /content/my_dataset
train: images
val: images

names:
"""
for idx, name in coco_names.items():
    yaml_content += f"  {idx}: {name}\n"

with open("/content/my_dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("Created /content/my_dataset/data.yaml")
print(f"Images: {IMG_DIR} ({len(image_files)} files)")
print(f"Labels: {LABEL_DIR} ({len(list(LABEL_DIR.glob('*.txt')))} files)")

---
# Part 3: Benchmarking

## 3.1 Latency Benchmark

In [ ]:
import time

def benchmark_model(model_path, image_path, use_rtdetr=False, warmup=10, runs=50):
    model = RTDETR(model_path) if use_rtdetr else YOLO(model_path)
    for _ in range(warmup):
        model(image_path, verbose=False)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    latencies = []
    for _ in range(runs):
        start = time.perf_counter()
        model(image_path, verbose=False)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        latencies.append((time.perf_counter()-start)*1000)
    latencies = np.array(latencies)
    return {"mean_ms": latencies.mean(), "std_ms": latencies.std(),
            "min_ms": latencies.min(), "max_ms": latencies.max(), "fps": 1000/latencies.mean()}

configs = [
    ("YOLOv8", "PyTorch", model_paths["yolov8m_pytorch"], False),
    ("YOLOv8", "ONNX", model_paths["yolov8m_onnx"], False),
    ("YOLOv8", "TensorRT", model_paths["yolov8m_tensorrt"], False),
    ("RT-DETR", "PyTorch", model_paths["rtdetr_l_pytorch"], True),
    ("RT-DETR", "ONNX", model_paths["rtdetr_l_onnx"], True),
    ("RT-DETR", "TensorRT", model_paths["rtdetr_l_tensorrt"], True),
]

latency_results = []
for model_name, backend, path, is_rtdetr in configs:
    print(f"Benchmarking {model_name}/{backend}...", end=" ", flush=True)
    result = benchmark_model(path, sample_path, use_rtdetr=is_rtdetr)
    result["model"] = model_name
    result["backend"] = backend
    latency_results.append(result)
    print(f"{result['mean_ms']:.1f} ms ({result['fps']:.1f} FPS)")
print("Done!")

In [ ]:
latency_df = pd.DataFrame(latency_results)[["model","backend","mean_ms","std_ms","min_ms","max_ms","fps"]]
latency_df.columns = ["Model","Backend","Mean (ms)","Std (ms)","Min (ms)","Max (ms)","FPS"]
for col in latency_df.columns[2:]: latency_df[col] = latency_df[col].round(1)
print("Latency Benchmark Results (T4 GPU, 640x640):")
print(latency_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
backends = ["PyTorch", "ONNX", "TensorRT"]
x = np.arange(3); w = 0.35
for ax, metric, title in [(axes[0], "mean_ms", "Latency (ms)"), (axes[1], "fps", "FPS")]:
    yv = [r[metric] for r in latency_results if r["model"]=="YOLOv8"]
    rv = [r[metric] for r in latency_results if r["model"]=="RT-DETR"]
    ax.bar(x-w/2, yv, w, label="YOLOv8", color="#2196F3")
    ax.bar(x+w/2, rv, w, label="RT-DETR", color="#FF9800")
    ax.set(xlabel="Backend", ylabel=title, title=title)
    ax.set_xticks(x, backends); ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("latency_benchmark.png", dpi=150); plt.show()

## 3.2 Accuracy (mAP) on Our Own Annotations

In [ ]:
def evaluate_map(model_path, use_rtdetr=False):
    model = RTDETR(model_path) if use_rtdetr else YOLO(model_path)
    results = model.val(data="/content/my_dataset/data.yaml", verbose=False)
    return {"map50": results.box.map50, "map50_95": results.box.map}

map_results = []
for model_name, backend, path, is_rtdetr in configs:
    print(f"mAP: {model_name}/{backend}...", end=" ", flush=True)
    result = evaluate_map(path, use_rtdetr=is_rtdetr)
    result["model"] = model_name
    result["backend"] = backend
    map_results.append(result)
    print(f"mAP@50: {result['map50']:.3f}, mAP@50:95: {result['map50_95']:.3f}")
print("Done!")

## 3.3 Combined Results Table

In [ ]:
combined = []
for lat, acc in zip(latency_results, map_results):
    combined.append({"Model": lat["model"], "Backend": lat["backend"],
        "mAP@50": round(acc["map50"],3), "mAP@50:95": round(acc["map50_95"],3),
        "Avg Latency (ms)": round(lat["mean_ms"],1), "FPS": round(lat["fps"],1)})
combined_df = pd.DataFrame(combined)
print("\nFinal Results — Our Own Annotated Data:")
print(combined_df.to_string(index=False))
combined_df.to_csv("benchmark_results.csv", index=False)
print("\nSaved to benchmark_results.csv")

---
# Part 4: Video Inference Demo

Run all 6 model/backend combos on our video to demonstrate inference on video data.

In [ ]:
if VIDEO_PATH and Path(VIDEO_PATH).exists():
    cap = cv2.VideoCapture(VIDEO_PATH)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    print(f"Video: {VIDEO_PATH}")
    print(f"Frames: {total_frames}, FPS: {fps:.1f}, Duration: {total_frames/fps:.1f}s")
else:
    print("No video found. Upload a .mp4 to your Drive folder.")

In [ ]:
# Run each model/backend on the video (first 30 frames)
if VIDEO_PATH:
    MAX_FRAMES = 30
    video_results = []
    
    for model_name, backend, path, is_rtdetr in configs:
        model = RTDETR(path) if is_rtdetr else YOLO(path)
        cap = cv2.VideoCapture(VIDEO_PATH)
        
        frame_times = []
        frame_idx = 0
        while cap.isOpened() and frame_idx < MAX_FRAMES:
            ret, frame = cap.read()
            if not ret: break
            start = time.perf_counter()
            model(frame, verbose=False)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            frame_times.append((time.perf_counter()-start)*1000)
            frame_idx += 1
        cap.release()
        
        avg_ms = np.mean(frame_times)
        video_results.append({"Model": model_name, "Backend": backend,
            "Frames": frame_idx, "Avg ms/frame": round(avg_ms,1), "Video FPS": round(1000/avg_ms,1)})
        print(f"{model_name}/{backend}: {avg_ms:.1f} ms/frame ({1000/avg_ms:.1f} FPS)")
    
    video_df = pd.DataFrame(video_results)
    print("\nVideo Inference Results:")
    print(video_df.to_string(index=False))

In [ ]:
# Show annotated frames from the video (using fastest model)
if VIDEO_PATH:
    best_model = YOLO(model_paths["yolov8m_tensorrt"])
    cap = cv2.VideoCapture(VIDEO_PATH)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    frame_indices = [0, 5, 10, 15, 20, 25]
    for i, target_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
        ret, frame = cap.read()
        if not ret: break
        results = best_model(frame, verbose=False)
        annotated = results[0].plot()
        axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"Frame {target_idx}", fontsize=10)
        axes[i].axis("off")
    
    cap.release()
    plt.suptitle("Video Detection — YOLOv8 TensorRT", fontsize=13)
    plt.tight_layout()
    plt.savefig("video_detections.png", dpi=150)
    plt.show()

---
# Part 5: Backend Server

```
React (localhost:5173) --> ngrok --> Colab FastAPI (:8000) --> GPU inference
```

In [ ]:
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- REPLACE THIS
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
%%writefile main.py
import time, json, tempfile
from pathlib import Path
import cv2, numpy as np
from fastapi import FastAPI, File, UploadFile, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO, RTDETR

app = FastAPI(title="Object Detection API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

with open("model_paths.json") as f:
    model_paths = json.load(f)
models = {}
for key, path in model_paths.items():
    if Path(path).exists():
        print(f"Loading {key}...")
        models[key] = RTDETR(path) if "rtdetr" in key else YOLO(path)
print(f"Loaded {len(models)} models")

@app.get("/health")
def health(): return {"status": "ok", "models_loaded": list(models.keys())}

@app.get("/models")
def list_models(): return {"models": [{"key": k, "path": model_paths.get(k,"")} for k in models]}

@app.post("/detect")
async def detect(file: UploadFile = File(...), model: str = Query(default="yolov8m_pytorch"), confidence: float = Query(default=0.25, ge=0.0, le=1.0)):
    if model not in models: return JSONResponse(status_code=400, content={"error": f"Unknown model: {model}"})
    img = cv2.imdecode(np.frombuffer(await file.read(), np.uint8), cv2.IMREAD_COLOR)
    if img is None: return JSONResponse(status_code=400, content={"error": "Could not decode image"})
    start = time.perf_counter()
    results = models[model](img, conf=confidence, verbose=False)
    latency_ms = (time.perf_counter()-start)*1000
    detections = []
    for box in results[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        detections.append({"bbox":[round(x1,1),round(y1,1),round(x2,1),round(y2,1)], "confidence":round(float(box.conf[0]),3), "class_id":int(box.cls[0]), "class_name":results[0].names[int(box.cls[0])]})
    return {"detections": detections, "model": model, "latency_ms": round(latency_ms,1), "image_size": {"width":img.shape[1],"height":img.shape[0]}}

@app.post("/detect-video")
async def detect_video(file: UploadFile = File(...), model: str = Query(default="yolov8m_pytorch"), confidence: float = Query(default=0.25, ge=0.0, le=1.0), max_frames: int = Query(default=30, ge=1, le=300)):
    if model not in models: return JSONResponse(status_code=400, content={"error": f"Unknown model: {model}"})
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        tmp.write(await file.read()); tmp_path = tmp.name
    cap = cv2.VideoCapture(tmp_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30; total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_results, frame_idx, total_latency = [], 0, 0
    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret: break
        start = time.perf_counter()
        results = models[model](frame, conf=confidence, verbose=False)
        latency_ms = (time.perf_counter()-start)*1000; total_latency += latency_ms
        dets = []
        for box in results[0].boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            dets.append({"bbox":[round(x1,1),round(y1,1),round(x2,1),round(y2,1)], "confidence":round(float(box.conf[0]),3), "class_id":int(box.cls[0]), "class_name":results[0].names[int(box.cls[0])]})
        frames_results.append({"frame":frame_idx, "detections":dets, "latency_ms":round(latency_ms,1)})
        frame_idx += 1
    cap.release(); Path(tmp_path).unlink(missing_ok=True)
    return {"frames":frames_results, "model":model, "video_fps":round(fps,1), "total_frames_processed":frame_idx, "total_frames_in_video":total_frames, "avg_latency_ms":round(total_latency/max(frame_idx,1),1)}

In [ ]:
public_url = ngrok.connect(8000)
print(f"\nBackend URL: {public_url}")
print(f"Docs: {public_url}/docs")

In [ ]:
import uvicorn
uvicorn.run("main:app", host="0.0.0.0", port=8000)